In [2]:
from bob.core import (
    bind_model_namespace,
    dump,
    turtle,
    ExternalReference,
    get_datagraph,
    Value,
    quantitykind,
    unit,
    enum,
    Device, 
    Junction
)
from rdflib import URIRef

from bob.devices.electricity.distribution import DistributionPanel, CircuitBreaker
from bob.connections.electricity import (
    Electricity_120V_60HzInletConnectionPoint,
    Electricity_120V_60HzOutletConnectionPoint,
    Electricity_240V_60HzInletConnectionPoint,
    Electricity_240V_60HzOutletConnectionPoint,
    Electricity_120V_240V_60HzInletConnectionPoint,
)
from bob.space.hvac import HVACSpace, HVACZone
from bob.space.physical import Building, Floor, Office, MechanicalRoom
from bob.core import DomainSpace, HVAC, PhysicalSpace
from bob.devices.hvac.fan import Fan
from bob.connections.air import AirConnection
__namespace__ = bind_model_namespace("ex", "urn:ex/")

In [3]:

building = Building(label="My Building")
floor = Floor(label="Floor1")
basement = Floor(label="basement")
office1 = Office(label="Office 1")
office2 = Office(label="Office 2")
office3 = Office(label="Office 3")
mechroom = MechanicalRoom(label="Mechanical Room")

office1_hvac = HVACSpace(label="Office 1")
office2_hvac = HVACSpace(label="Office 2")
office3_hvac = HVACSpace(label="Office 3")

zone1 = HVACZone(label="Zone1")

building > floor
building > basement
floor > office1
floor > office2
floor > office3
basement > mechroom
office1_hvac < office1
office2_hvac < office2
office3_hvac < office3

office1_hvac < zone1
office2_hvac < zone1

sf = Fan(label="Supply Fan", hasPhysicalLocation=mechroom)
rf = Fan(label="Return Fan", hasPhysicalLocation=mechroom)
# Here we make it a junction but it would be better to be a Simple Connection... it's for test purposes
supply_duct = Junction(label="J1", hasSubstance=enum["Medium-Air"])
supply_duct.link_to(sf.airOutlet)
supply_duct >> office1_hvac.airInlet
supply_duct >> office2_hvac.airInlet
return_plenum = AirConnection(
    label="RETURN-AIR", comment="Air returns from zone here"
)
# return_plenum = Junction(label="J1", hasSubstance=enum['Medium-Air'])
# return_plenum.link_to(_returnAir.airInlet)
office1_hvac.airOutlet >> return_plenum
office2_hvac.airOutlet >> return_plenum
return_plenum >> rf

# and the zone ?
zone1.airInlet.maps_to = supply_duct
zone1.airOutlet.maps_to = return_plenum

In [3]:
f = Fan(label='Fan')
supply_duct = Junction(label="J1", hasSubstance=enum['Medium-Air'])
supply_duct.link_to(f.airOutlet)
supply_duct >> office1_hvac.airInlet
supply_duct >> office2_hvac.airInlet
return_plenum = AirConnection(label="RETURN-AIR", comment="Air returns from zone here")
#return_plenum = Junction(label="J1", hasSubstance=enum['Medium-Air'])
#return_plenum.link_to(_returnAir.airInlet)
office1_hvac.airOutlet >> return_plenum
office2_hvac.airOutlet >> return_plenum

{'node': rdflib.term.URIRef('urn:ex/00024'), 'label': 'RETURN-AIR', 'comment': 'Air returns from zone here', 'hasSubstance': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Medium-Air')}

In [11]:
j.link_to(f.airOutlet)

In [19]:
zone1.airInlet

AttributeError: 'HVACZone' object has no attribute 'airInlet'

In [18]:
dump()

@prefix enum: <http://data.ashrae.org/standard223/1.0/vocab/enumeration#> .
@prefix ex: <urn:ex/> .
@prefix quantitykind: <http://qudt.org/vocab/quantitykind/> .
@prefix qudt: <http://qudt.org/schema/qudt/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .
@prefix unit: <http://qudt.org/vocab/unit/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:00001 a s223:PhysicalSpace ;
    rdfs:label "My Building" ;
    s223:contains ex:00002 .

ex:00008 a s223:DomainSpace,
        s223:HVACSpace ;
    rdfs:label "Office 3" ;
    s223:hasDomain s223:HVAC .

ex:00009 a s223:HVACZone,
        s223:Zone ;
    rdfs:label "Zone1" ;
    s223:contains ex:00006,
        ex:00007 ;
    s223:hasDomain s223:HVAC .

ex:00010 a s223:Junction ;
    rdfs:label "J1" .

ex:00011 a s223:Device,
        s223:Fan ;
    rdfs:label "Fan" ;
    s223:airInlet ex:00012 ;
    s223:airOutlet ex:00013 ;
    s223:electricalInlet ex:00014 ;
    s223:hasCon

In [2]:
distributionpanel_template = {
        "device": {
            "label": "Name Of Panel",
            "comment": "Description",
            "electricalInlet": Electricity_120V_240V_60HzInletConnectionPoint,
            "electricalOutletA": Electricity_120V_60HzOutletConnectionPoint,
            "electricalOutletB": Electricity_240V_60HzOutletConnectionPoint,
            # "hasMeasurementLocation": Connection,
        },
        "sensors": {},
        "contains": {
            ("CircBreaker_120_#1", CircuitBreaker): {
                "comment": "Office lights #1",
                "hasSubstance": enum["Electricity-120V.60Hz"],
                "electricalInlet": Electricity_120V_60HzInletConnectionPoint,
                "electricalOutlet": Electricity_120V_60HzOutletConnectionPoint,
                "hasMaxRange": Value(
                    15, hasQuantityKind=quantitykind.ElectricCurrent, unit=unit.A
                ),
            },
            ("CircBreaker_240_#2", CircuitBreaker): {
                "comment": "Heating Room #1",
                "hasSubstance": enum["Electricity-240V.60Hz"],
                "electricalInlet": Electricity_240V_60HzInletConnectionPoint,
                "electricalOutlet": Electricity_240V_60HzOutletConnectionPoint,
                "hasMaxRange": Value(
                    20, hasQuantityKind=quantitykind.ElectricCurrent, unit=unit.A
                ),
            },
        },
        # other properties could go there... ?
    }

In [3]:
cb = CircuitBreaker(label='CB1',
                    comment="Heating Room #1",
                    hasSubstance= enum["Electricity-240V.60Hz"],
                    electricalInlet= Electricity_240V_60HzInletConnectionPoint,
                    electricalOutlet= Electricity_240V_60HzOutletConnectionPoint,
                    hasMaxRange= Value(
                        20, hasQuantityKind=quantitykind.ElectricCurrent, unit=unit.A
                    ))

cb2 = CircuitBreaker(label='CB2',
                    comment="Heating Room #2",
                    hasSubstance= enum["Electricity-240V.60Hz"],
                    electricalInlet= Electricity_240V_60HzInletConnectionPoint,
                    electricalOutlet= Electricity_240V_60HzOutletConnectionPoint,
                    hasMaxRange= Value(
                        20, hasQuantityKind=quantitykind.ElectricCurrent, unit=unit.A
                    ))



In [4]:
cb >> cb2

{'node': rdflib.term.URIRef('urn:ex/00008'), 'label': 'CB2', 'comment': 'Heating Room #2', 'hasRole': None, 'hasLocation': None, 'hasSubstance': {'node': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Electricity-240V.60Hz'), 'label': '', 'comment': ''}, 'hasMaxRange': {'unit': rdflib.term.URIRef('http://qudt.org/vocab/unit/A'), 'hasQuantityKind': rdflib.term.URIRef('http://qudt.org/vocab/quantitykind/ElectricCurrent'), 'node': rdflib.term.URIRef('urn:ex/00007'), 'label': '', 'comment': '', 'isValueOf': None, 'hasTimestamp': None, 'hasSimpleValue': rdflib.term.Literal('20', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))}, '_connection_points': {'urn:ex/00009': {'node': rdflib.term.URIRef('urn:ex/00009'), 'label': 'CB2.electricalInlet', 'comment': '', 'hasSubstance': {'node': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Electricity-240V.60Hz'), 'label': '', 'comment': ''}, 'hasDirection': {'node': rd

In [5]:
panel = DistributionPanel(label= "Name Of Panel",
                    comment= "Description",
                    electricalInlet= Electricity_120V_240V_60HzInletConnectionPoint,
                    electricalOutletA= Electricity_120V_60HzOutletConnectionPoint,
                    electricalOutletB= Electricity_240V_60HzOutletConnectionPoint)

In [6]:
panel >> cb

{'node': rdflib.term.URIRef('urn:ex/00004'), 'label': 'CB1', 'comment': 'Heating Room #1', 'hasRole': None, 'hasLocation': None, 'hasSubstance': {'node': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Electricity-240V.60Hz'), 'label': '', 'comment': ''}, 'hasMaxRange': {'unit': rdflib.term.URIRef('http://qudt.org/vocab/unit/A'), 'hasQuantityKind': rdflib.term.URIRef('http://qudt.org/vocab/quantitykind/ElectricCurrent'), 'node': rdflib.term.URIRef('urn:ex/00003'), 'label': '', 'comment': '', 'isValueOf': None, 'hasTimestamp': None, 'hasSimpleValue': rdflib.term.Literal('20', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))}, '_connection_points': {'urn:ex/00005': {'node': rdflib.term.URIRef('urn:ex/00005'), 'label': 'CB1.electricalInlet', 'comment': '', 'hasSubstance': {'node': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Electricity-240V.60Hz'), 'label': '', 'comment': ''}, 'hasDirection': {'node': rd

In [15]:
cwc.label

'a'

NameError: name 'panel' is not defined

In [18]:
from bob.core import connect,  Connection, OutletConnectionPoint, Connectable, ConnectionPoint, InletConnectionPoint
from rdflib import URIRef
from collections import defaultdict
from typing import Dict, Optional, Set, Any, TextIO, Tuple, TypeVar, Union, cast

#connect(panel, cb)

In [7]:
from_types: Set[URIRef]
from_thing = panel 
to_thing = cb

In [8]:
isinstance(from_thing, Connectable)

True

In [9]:
from_out = defaultdict(set)
for attr, connection_point in from_thing._connection_points.items():
    if connection_point.connectsThrough:
        continue
    if not isinstance(connection_point, OutletConnectionPoint):
        continue

    medium = getattr(connection_point, "hasSubstance", None)
    medium = getattr(medium, "node", medium)
    from_out[medium].add(connection_point)


In [13]:
from_out

defaultdict(set,
            {rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Electricity-120V.60Hz'): {{'node': rdflib.term.URIRef('urn:ex/00013'), 'label': 'Name Of Panel.electricalOutletA', 'comment': '', 'hasSubstance': {'node': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Electricity-120V.60Hz'), 'label': '', 'comment': ''}, 'hasDirection': {'node': rdflib.term.URIRef('http://data.ashrae.org/standard223#Outlet'), 'label': '', 'comment': ''}, 'lnx': None, 'connectsThrough': None, 'isConnectionPointOf': {'node': rdflib.term.URIRef('urn:ex/00011'), 'label': 'Name Of Panel', 'comment': 'Description', 'hasRole': None, 'hasLocation': None, '_connection_points': {'urn:ex/00012': {'node': rdflib.term.URIRef('urn:ex/00012'), 'label': 'Name Of Panel.electricalInlet', 'comment': '', 'hasSubstance': {'node': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Electricity-120V_240V.60Hz'), 'label': '', 'comme

In [11]:
from_types: Set[URIRef]
if isinstance(from_thing, Connection):
    from_types = set([from_thing.hasSubstance])
else:
    from_types = set(medium for medium in from_out if len(from_out[medium]) == 1)
    if not from_types:
        from_types = set(medium for medium in from_out if len(from_out[medium.node]) == 1)
    if not from_types:
        raise RuntimeError(f"no candidate sources from {from_thing} to {to_thing}")


In [12]:
from_types

{rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Electricity-120V.60Hz'),
 rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Electricity-240V.60Hz')}

In [19]:
to_in = defaultdict(set)
if isinstance(to_thing, Connectable):
    for attr, connection_point in to_thing._connection_points.items():
        if connection_point.connectsThrough:
            continue
        if not isinstance(connection_point, InletConnectionPoint):
            continue

        medium = getattr(connection_point, "hasSubstance", None)
        # Here when trying to connect a connectionpoint to a device
        # medium turned to be
        # {'node': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Water-ChilledWater'), 'label': '', 'comment': ''}
        # and the intersection fails to recognize the substance
        medium = getattr(medium, "node", medium)
        to_in[medium].add(connection_point)

In [45]:
len(to_in[medium])

2

In [48]:
print(to_in)

defaultdict(<class 'set'>, {rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Electricity-240V.60Hz'): {{'node': rdflib.term.URIRef('urn:ex/00006'), 'label': 'CB1.electricalOutlet', 'comment': '', 'hasSubstance': {'node': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Electricity-240V.60Hz'), 'label': '', 'comment': ''}, 'hasDirection': {'node': rdflib.term.URIRef('http://data.ashrae.org/standard223#Inlet'), 'label': '', 'comment': ''}, 'lnx': None, 'connectsThrough': None, 'isConnectionPointOf': {'node': rdflib.term.URIRef('urn:ex/00004'), 'label': 'CB1', 'comment': 'Heating Room #1', 'hasRole': None, 'hasLocation': None, 'hasSubstance': {'node': rdflib.term.URIRef('http://data.ashrae.org/standard223/1.0/vocab/enumeration#Electricity-240V.60Hz'), 'label': '', 'comment': ''}, 'hasMaxRange': {'unit': rdflib.term.URIRef('http://qudt.org/vocab/unit/A'), 'hasQuantityKind': rdflib.term.URIRef('http://qudt.org/vocab/quantitykind/Electr

In [31]:
to_types: Set[URIRef]
if isinstance(to_thing, Connection):
    to_types = set([to_thing.hasSubstance])
else:
    to_types = set(medium for medium in to_in if len(to_in[medium]) == 1)
    if not to_types:
        to_types = set(medium for medium in to_in if len(to_in[medium]) == 1)

In [29]:
to_types

set()

In [27]:
isinstance(to_thing, Connection)

False

In [9]:
from bob.core import bind_model_namespace, turtle, dump
from bob.core import Device, ConnectionPoint, Connection
from bob import core

__namespace__ = bind_model_namespace("ex", "urn:ex/")

core.EXPLICIT_RECIPROCITY = True


d1 = Device(label="d1")
cp1 = ConnectionPoint(d1)

c = Connection()
c.connect_to(cp1)

result = turtle()
dump()


def test_result():
    print(result)

@prefix ex: <urn:ex/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .

ex:00001 a s223:Device ;
    rdfs:label "d1" ;
    s223:hasConnectionPoint ex:00002 .

ex:00003 a s223:Connection ;
    s223:connectsAt ex:00002 .

ex:00002 a s223:ConnectionPoint ;
    s223:connectsThrough ex:00003 .

ex:00004 a s223:Device ;
    rdfs:label "d1" ;
    s223:connectedThrough ex:00006 ;
    s223:hasConnectionPoint ex:00005 .

ex:00005 a s223:ConnectionPoint ;
    s223:connectsThrough ex:00006 ;
    s223:isConnectionPointOf ex:00004 .

ex:00006 a s223:Connection ;
    s223:connectsAt ex:00005 ;
    s223:connectsTo ex:00004 .



In [5]:
import bob
print(bob.core.EXPLICIT_RECIPROCITY)

False


In [1]:
from bob.core import (
    bind_model_namespace,
    dump,
    turtle,
    ExternalReference,
    get_datagraph,
    Value,
    quantitykind,
    unit,
    enum,
    quantityValue,
)

from bob.devices.hvac.gas import GasMonitor
from bob.property import QuantifiableObservableProperty
from bob.sensor.gas import CO2Sensor, NO2Sensor, COSensor
from bob.sensor.temperature import AirTemperatureSensor
from bob.space.hvac import HVACSpace, HVACZone
from bob.space.physical import Building, Roof, Floor, Office

__namespace__ = bind_model_namespace("ex", "urn:ex/")

In [6]:


_config = {
    "sensors": {
        ("CO2_sensor", CO2Sensor): {
            "hasExternalReference": "bacnet://",
            "hasMinRange": QuantifiableObservableProperty(
                0,
                hasQuantityKind=quantitykind.DimensionlessRatio,
                unit=unit.PPM,
                label="CO2_sensor.MinRange",
            ),
            "hasMaxRange": QuantifiableObservableProperty(
                2000,
                hasQuantityKind=quantitykind.DimensionlessRatio,
                unit=unit.PPM,
                label="CO2_sensor.MaxRange",
            ),
        },
    }
    }
co2monitor = GasMonitor(
    label="CO2-4",
    comment="CO2 Monitor in basement",
    config=_config,
)
building = Building(label="My Building")
roof = Roof(label="Roof of building")
floor = Floor(label="Floor1")
basement = Floor(label="Basement")
office1 = Office(label="Office 1")
office2 = Office(label="Office 2")
office3 = Office(label="Office 3")
joelsoffice = Office(label="Joel's Office")

office1_hvac = HVACSpace(label="Office 1")
office2_hvac = HVACSpace(label="Office 2")
office3_hvac = HVACSpace(label="Office 3")
basementhvac = HVACSpace(label="Basement HVAC Space")

zone1 = HVACZone(label="Zone1")

# Physical relationships
building > roof
building > floor
building > basement > basementhvac
basement > joelsoffice
floor > office1
floor > office2
floor > office3


# Spaces relationships
# SPACES     | PHYSICAL
office1_hvac < office1
office2_hvac < office2
office3_hvac < office3

#basementhvac < basement

# Zones (group of spaces)
# Here, Zone1 contains office1 and office2
office1_hvac < zone1
office2_hvac < zone1

co2monitor.hasPhysicalLocation = basement
co2monitor['CO2_sensor'].hasMeasurementLocation = basementhvac


ARGS :  (0,)
ARGS :  (2000,)
ARGS :  ()


In [5]:
co2monitor._contains

[{'node': rdflib.term.URIRef('urn:ex/00085'), 'label': 'CO2_sensor', 'comment': '', 'hasRole': None, 'hasPhysicalLocation': None, 'hasMeasurementLocation': None, 'hasMeasurementPrecision': None, 'hasMeasurementUncertainty': None, 'hasMaxRange': {'unit': rdflib.term.URIRef('http://qudt.org/vocab/unit/PPM'), 'hasQuantityKind': rdflib.term.URIRef('http://qudt.org/vocab/quantitykind/DimensionlessRatio'), 'node': rdflib.term.URIRef('urn:ex/00084'), 'label': 'CO2_sensor.MaxRange', 'comment': '', 'hasValue': rdflib.term.Literal('2000', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#decimal')), 'hasExternalReference': None, 'isValueOf': None}, 'hasMinRange': {'unit': rdflib.term.URIRef('http://qudt.org/vocab/unit/PPM'), 'hasQuantityKind': rdflib.term.URIRef('http://qudt.org/vocab/quantitykind/DimensionlessRatio'), 'node': rdflib.term.URIRef('urn:ex/00083'), 'label': 'CO2_sensor.MinRange', 'comment': '', 'hasValue': rdflib.term.Literal('0', datatype=rdflib.term.URIRef('http://www